# wandb-watch-model — worked example 1: Watch a single linear layer for gradient histograms

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `wandb-watch-model`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

`wandb.watch(module, log='all', log_freq=N)` registers a module with wandb so that every `N` optimizer steps, wandb logs histograms of the module's parameters and their gradients. Using `log='all'` captures both; `log='gradients'` skips parameter histograms; `log='parameters'` skips gradient histograms. Calling `watch` on a targeted submodule (rather than the whole model) reduces dashboard noise.

## Worked solution

**Step 1 — identify the target module.**
We want to watch a specific linear layer, not the entire model. In this example the target is `model.classifier` — a single `nn.Linear` layer representing the output head.

**Step 2 — call wandb.watch.**
We call `wandb.watch(model.classifier, log='all', log_freq=log_freq)`. The first positional argument is the module to watch. `log='all'` means both parameter and gradient histograms will be recorded. `log_freq` controls how many optimizer steps pass between each histogram snapshot.

**Step 3 — return the watched module.**
Returning the watched module lets the caller confirm it got the right submodule (not an accidental reference to the whole model or a different layer).

In [ ]:
import sys
from unittest.mock import MagicMock
import torch.nn as nn
sys.modules.setdefault('wandb', MagicMock())
import wandb

class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = nn.Sequential(nn.Linear(128, 64), nn.ReLU())
        self.classifier = nn.Linear(64, 10)
    def forward(self, x):
        return self.classifier(self.backbone(x))

def watch_classifier(model, log_freq):
    """Watch only the classifier (output head) — not the whole model."""
    wandb.watch(model.classifier, log='all', log_freq=log_freq)
    return model.classifier

# Exercise it
wandb.watch.reset_mock()
model = SimpleModel()
watched = watch_classifier(model, log_freq=50)
print('Watched module:', watched)
call_args = wandb.watch.call_args
print('First arg is classifier:', call_args.args[0] is model.classifier)
print('log=all:', call_args.kwargs.get('log') == 'all')
print('log_freq:', call_args.kwargs.get('log_freq'))